# Process 파이프라인 테스트

`scripts/process/` 안의 운영 모듈(`hdfs_reader`, `mmsi_summary`, `cassandra_save`)을 직접 불러와 노트북에서 대화식으로 테스트합니다.

- HDFS 연결 확인
- 특정 경로의 JSON 레코드 스트리밍 확인
- MMSI별 집계 결과 확인 (Cassandra 저장 여부는 `SAVE_TO_CASSANDRA` 플래그로 제어)

In [4]:
from __future__ import annotations

import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

# scripts/notebook/process/ 에서 scripts/process/ 를 import 경로에 추가
PROCESS_DIR = Path.cwd().resolve().parents[1] / "process"
if str(PROCESS_DIR) not in sys.path:
    sys.path.insert(0, str(PROCESS_DIR))

import hdfs_reader
import mmsi_summary
import cassandra_save

## 1. HDFS 연결 확인

In [5]:
hdfs_reader.check_connection()

  [HDFS] 연결 성공 (http://namenode:9870)


True

## 2. 조회할 HDFS 경로 지정

분 단위 경로가 필요하면 `get_hdfs_path`, 시간 단위 전체를 읽고 싶으면 `get_hdfs_path_hour`를 사용합니다.

In [8]:
TARGET_DT = datetime(2026,5,1,0)
# TARGET_DT = datetime.now()  # 필요하면 datetime(2026, 3, 20, 12, 0) 처럼 직접 지정

hdfs_path = mmsi_summary.get_hdfs_path_hour(TARGET_DT)
print(f"조회 경로: {hdfs_path}")

조회 경로: /datalake/ais/2026/05/01/00


## 3. 레코드 스트리밍 미리보기

전체를 다 읽지 않고 앞부분 N건만 확인합니다.

In [9]:
PREVIEW_LIMIT = 5

preview = []
for record in hdfs_reader.stream_records(hdfs_path):
    preview.append(record)
    if len(preview) >= PREVIEW_LIMIT:
        break

preview

  [HDFS 읽기] /datalake/ais/2026/05/01/00/00/ais_20260501_000041.json


[{'messageName': '!AIVDM',
  'totalMessage': 1,
  'messageNumber': 1,
  'encodedData': '18IJqL?CQj9?@f@CqpiSH2ID0hFQ',
  'data': {'messageId': 1,
   'repeatIndicator': 0,
   'mmsi': 563526000,
   'navigationalStatus': 15,
   'rateOfTurn': 78,
   'sog': 11.4,
   'positionAccuracy': 0,
   'longitude': 129.163,
   'latitude': 34.78561,
   'cog': 86.4,
   'trueHeading': 76,
   'timeStamp': 42,
   'specialManoeuvreIndicator': 0,
   'spare': 0,
   'raimFlag': 0,
   'communcation state': {'syncState': 1,
    'slotTimeout': 4,
    'subMessage': 1441}},
  'channel': 'B',
  'sequenceId': '7',
  'vsi': {'slotNum': '1441', 'rssi': '-111.2', 'snr': '9.5'},
  'dataBucket': '2026-04-30-23:59:39.443',
  'stationMmsi': '004403102'},
 {'messageName': '!AIVDM',
  'totalMessage': 1,
  'messageNumber': 1,
  'encodedData': '15AUe`>lDC9>bfHD10LcQ2ID0`GQ',
  'data': {'messageId': 1,
   'repeatIndicator': 0,
   'mmsi': 353988000,
   'navigationalStatus': 14,
   'rateOfTurn': -47,
   'sog': 27.5,
   'positionAc

## 4. MMSI별 집계 실행

`SAVE_TO_CASSANDRA = True`로 바꾸면 읽으면서 동시에 `cassandra_save.save_record()`로 저장합니다.
테스트 단계에서는 `False`로 두고 집계 결과만 확인하는 것을 권장합니다.

In [10]:
SAVE_TO_CASSANDRA = False

df = mmsi_summary.build_summary(hdfs_path, save=SAVE_TO_CASSANDRA)

if SAVE_TO_CASSANDRA:
    cassandra_save.close()

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)
df

  [HDFS 읽기] /datalake/ais/2026/05/01/00/00/ais_20260501_000041.json
  [HDFS 읽기] /datalake/ais/2026/05/01/00/00/ais_20260501_000142.json
  [HDFS 읽기] /datalake/ais/2026/05/01/00/00/ais_20260501_000243.json
  [HDFS 읽기] /datalake/ais/2026/05/01/00/00/ais_20260501_000344.json
  [HDFS 읽기] /datalake/ais/2026/05/01/00/00/ais_20260501_000445.json
  [HDFS 읽기] /datalake/ais/2026/05/01/00/00/ais_20260501_000546.json
  [HDFS 읽기] /datalake/ais/2026/05/01/00/00/ais_20260501_000647.json
  [HDFS 읽기] /datalake/ais/2026/05/01/00/00/ais_20260501_000748.json
  [HDFS 읽기] /datalake/ais/2026/05/01/00/00/ais_20260501_000849.json
  [HDFS 읽기] /datalake/ais/2026/05/01/00/00/ais_20260501_000950.json
  [HDFS 읽기] /datalake/ais/2026/05/01/00/10/ais_20260501_001051.json
  [HDFS 읽기] /datalake/ais/2026/05/01/00/10/ais_20260501_001152.json
  [HDFS 읽기] /datalake/ais/2026/05/01/00/10/ais_20260501_001253.json
  [HDFS 읽기] /datalake/ais/2026/05/01/00/10/ais_20260501_001354.json
  [HDFS 읽기] /datalake/ais/2026/05/01/00/10/ais_2

,mmsi,startTime,endTime,count
0,440784000,2026-04-30 23:59:39.803,2026-05-01 00:59:41.798,1812
1,538008635,2026-04-30 23:59:40.769,2026-05-01 00:59:40.771,1810
2,215931000,2026-04-30 23:59:39.459,2026-05-01 00:59:41.472,1809
3,431224000,2026-04-30 23:59:39.644,2026-05-01 00:59:41.642,1808
4,636024658,2026-04-30 23:59:40.301,2026-05-01 00:59:40.305,1808
5,256730000,2026-04-30 23:59:40.160,2026-05-01 00:59:40.162,1806
6,255806421,2026-04-30 23:59:39.676,2026-05-01 00:59:41.672,1804
7,353988000,2026-04-30 23:59:40.206,2026-05-01 00:59:40.194,1800
8,538008815,2026-04-30 23:59:41.082,2026-05-01 00:59:41.083,1797
9,352002511,2026-04-30 23:59:41.081,2026-05-01 00:59:41.083,1797
